In [0]:
import polars as pl

# Krok 1: Załaduj dane ze Spark do Polars
print("Loading data...")

# Wczytaj tylko potrzebne kolumny, aby uniknąć problemów z konwersją dat
funding_spark = spark.table("unocha.default.fts_incoming_funding_global").select(
    "amountUSD", "boundary", "destLocations"
)
requirements_spark = spark.table("unocha.default.fts_requirements_funding_global").select(
    "countryCode", "requirements"
)

# Konwertuj do Polars
funding_df = pl.from_pandas(funding_spark.toPandas())
requirements_df = pl.from_pandas(requirements_spark.toPandas())

print(f"Funding records: {len(funding_df):,}")
print(f"Requirements records: {len(requirements_df):,}")

In [0]:
# Krok 2: Filtrowanie - tylko boundary = 'incoming'
print("\nFiltering incoming funding...")
incoming_df = funding_df.filter(pl.col("boundary") == "incoming")
print(f"Incoming funding records: {len(incoming_df):,}")

# Krok 3: Rozwijanie destLocations (split po przecinku) i agregacja
print("\nExploding country codes and aggregating...")

# Rozbij destLocations na pojedyncze kody krajów
incoming_exploded = incoming_df.with_columns(
    pl.col("destLocations").str.split(",").alias("country_codes")
).explode("country_codes")

# Agreguj sumy po kodzie kraju
aggregated_funding = (
    incoming_exploded
    .group_by("country_codes")
    .agg(
        pl.col("amountUSD").sum().alias("total_incoming_usd")
    )
    .filter(pl.col("country_codes").is_not_null())
)

print(f"Unique countries with funding: {len(aggregated_funding)}")
display(aggregated_funding.head(10))

In [0]:
# Krok 4: Przygotuj dane requirements (agregacja po kraju)
print("\nPreparing requirements data...")
requirements_by_country = (
    requirements_df
    .group_by("countryCode")
    .agg(
        pl.col("requirements").sum().alias("requirementsUSD")
    )
    .filter(pl.col("requirementsUSD").is_not_null())
)

print(f"Countries with requirements: {len(requirements_by_country)}")

# Krok 5: Łączenie - left join requirements z funding
print("\nJoining requirements with funding...")
gap_analysis = requirements_by_country.join(
    aggregated_funding,
    left_on="countryCode",
    right_on="country_codes",
    how="left"
)

# Krok 6: Obliczenia
print("\nCalculating coverage ratio and funding gap...")
gap_analysis = gap_analysis.with_columns([
    # Zastąp null zerami w total_incoming_usd
    pl.col("total_incoming_usd").fill_null(0).alias("total_incoming_usd"),
]).with_columns([
    # Oblicz coverage_ratio
    (pl.col("total_incoming_usd") / pl.col("requirementsUSD")).alias("coverage_ratio"),
    # Oblicz funding_gap
    (pl.col("requirementsUSD") - pl.col("total_incoming_usd")).alias("funding_gap")
])

print(f"\nTotal countries in analysis: {len(gap_analysis)}")

In [0]:
# Krok 7: Ranking - sortuj po coverage_ratio rosnąco
print("\n" + "="*60)
print("TOP 10 NAJBARDZIEJ POMINIĘTYCH KRYZYSÓW HUMANITARNYCH")
print("(posortowane według najniższego wskaźnika pokrycia)")
print("="*60)

most_overlooked = (
    gap_analysis
    .sort("coverage_ratio")
    .select([
        "countryCode",
        "requirementsUSD",
        "total_incoming_usd",
        "coverage_ratio",
        "funding_gap"
    ])
    .head(10)
)

# Formatuj wyniki dla lepszej czytelności
most_overlooked_formatted = most_overlooked.with_columns([
    (pl.col("coverage_ratio") * 100).round(2).alias("coverage_pct"),
    (pl.col("requirementsUSD") / 1_000_000).round(2).alias("requirements_mln_usd"),
    (pl.col("total_incoming_usd") / 1_000_000).round(2).alias("incoming_mln_usd"),
    (pl.col("funding_gap") / 1_000_000).round(2).alias("gap_mln_usd")
]).select([
    "countryCode",
    "requirements_mln_usd",
    "incoming_mln_usd",
    "gap_mln_usd",
    "coverage_pct"
])

print("\nWyniki (kwoty w milionach USD):\n")
display(most_overlooked_formatted)

print(f"\nℹ️ Wskaźnik pokrycia (coverage_pct) pokazuje % pokrycia potrzeb finansowych.")
print(f"   Niższy wskaźnik = bardziej pominięty kryzys.")